In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv("all_participants_data.csv", index_col="Participant")

indexes = data.index

#getting list of median arousals and valences

allMedianArousals = list(data["median_arousal"])
allMedianValences = list(data["median_valence"])

#getting median of the rows

medianArousal = data["median_arousal"].median()
medianValence = data["median_valence"].median()

data = data.drop(columns=['median_arousal', 'median_valence'])

scaler = StandardScaler()
scaledArray = scaler.fit_transform(data)
scaledData = pd.DataFrame(scaledArray, index=indexes)

print(scaledData)


                  0         1         2         3         4         5    \
Participant                                                               
64          -1.665550 -1.018178  0.162946  0.311937 -0.094151 -1.458478   
64          -1.547531 -0.988172  0.422245  0.623738 -0.008691 -1.384106   
64          -0.977216 -0.724075 -0.146381  0.229324  0.442454 -1.105813   
64          -0.218558  0.150993  0.458450  0.784200  1.168580 -0.211696   
64           0.075725  0.221303  0.027481  0.288399  0.747573 -0.061314   
...               ...       ...       ...       ...       ...       ...   
30           1.063190  1.484203  0.569686  0.643383  1.267968  1.515181   
30           0.939504  1.424867  1.099175  1.180893  1.795443  1.487396   
30           0.976282  1.368941  1.408992  1.329795  2.022155  1.450898   
30           1.191407  1.334019  1.951544  1.708168  2.398909  1.506098   
30           1.280602  1.261865  0.550702  0.383950  1.952202  1.441351   

                  6     

In [3]:
#getting lists of low and high arousals and valences

lowArousal = [(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] <= medianArousal]
highArousal = [(row, 1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] > medianArousal]

lowValence = [(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] <= medianValence]
highValence = [(row, 1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] > medianValence]

#getting percentile classes for arousals and valences

arousalPercentiles = np.percentile(allMedianArousals, [20, 40, 60, 80])
arousalPercentileClasses = []

arousalPercentileClasses.append([(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] <= arousalPercentiles[0]])
for i in range(0, 3):
    arousalPercentileClasses.append([(row, i+1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] > arousalPercentiles[i] and allMedianArousals[j] <= arousalPercentiles[i+1]])
arousalPercentileClasses.append([(row, 4, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] > arousalPercentiles[3]])

arousalPercentileClasses = sum(arousalPercentileClasses, [])


valencePercentiles = np.percentile(allMedianValences, [20, 40, 60, 80])
valencePercentileClasses = []

valencePercentileClasses.append([(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] <= valencePercentiles[0]])
for i in range(0, 3):
    valencePercentileClasses.append([(row, i+1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] > valencePercentiles[i] and allMedianValences[j] <= valencePercentiles[i+1]])
valencePercentileClasses.append([(row, 4, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] > valencePercentiles[3]])

valencePercentileClasses = sum(valencePercentileClasses, [])

In [4]:
from sklearn.linear_model import LogisticRegression

model1 = LogisticRegression(max_iter=10000)

In [5]:
from sklearn.neural_network import MLPClassifier

model2 = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', solver='adam', max_iter=1000, random_state=42)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=288,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)     
        last_out = lstm_out[:, -1, :]  
        out = self.fc(last_out)        
        return out
    
model3 = LSTMClassifier(1, 16, 2, 2)

In [7]:
from scipy.stats import pearsonr
import numpy as np

def CCcoefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

In [ ]:
# testing model 1

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in valencePercentileClasses]
Y = [element[1] for element in valencePercentileClasses]
orderedIndexes = [element[2] for element in valencePercentileClasses]

results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model1.fit(Xtrain, Ytrain)

    Ypred = model1.predict_proba(Xtest)
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

[0      0.538877
1      0.792790
2     -0.038362
3     -0.092644
4     -1.025014
         ...   
283   -0.778811
284   -0.158129
285    0.914081
286    0.804085
287   -0.879332
Name: 64, Length: 288, dtype: float64, 0      0.314105
1      0.754065
2     -0.631779
3     -0.615501
4     -1.329554
         ...   
283   -0.226745
284   -0.072710
285    0.055563
286    0.671908
287   -2.000556
Name: 64, Length: 288, dtype: float64, 0      0.218331
1      0.772767
2     -0.598048
3     -0.510164
4     -0.893755
         ...   
283    1.060220
284    0.057805
285   -0.287721
286    0.755685
287   -0.000641
Name: 64, Length: 288, dtype: float64, 0      0.202776
1      0.770577
2     -0.350122
3     -0.322829
4     -0.347474
         ...   
283   -0.388086
284   -0.164081
285    0.031543
286    0.308412
287   -1.727494
Name: 64, Length: 288, dtype: float64, 0      0.242372
1      0.816937
2     -0.457918
3     -0.413089
4     -0.558738
         ...   
283   -0.466677
284   -0.217265
285   -0.16

In [ ]:
# testing model 2

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in valencePercentileClasses]
Y = [element[1] for element in valencePercentileClasses]
orderedIndexes = [element[2] for element in valencePercentileClasses]

results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model2.fit(Xtrain, Ytrain)

    Ypred = model2.predict_proba(Xtest)
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

C:\Users\james\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [8]:
# testing model 3

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in arousalPercentileClasses]
Y = [element[1] for element in arousalPercentileClasses]
orderedIndexes = [element[2] for element in arousalPercentileClasses]

results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    Xtrain, Ytrain = torch.tensor(Xtrain, dtype = torch.float).unsqueeze(1), torch.tensor(Ytrain, dtype = torch.long)
    Xtest, Ytest = torch.tensor(Xtest, dtype = torch.float).unsqueeze(1), torch.tensor(Ytest, dtype = torch.float).unsqueeze(1)

    optimizer = optim.Adam(model3.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(10):
        optimizer.zero_grad()
        outputs = model3(Xtrain)
        loss = criterion(outputs, Ytrain)
        loss.backward()
        optimizer.step()

    model3.eval()
    with torch.no_grad():
        probs = torch.softmax(model3(Xtest), dim=1)
        expected = np.dot(probs, np.arange(5))

        probs = probs.argmax(axis=1)




        results.append((pearsonr(np.array(probs).ravel(), np.array(expected).ravel())[0], CCcoefficient(probs, expected)))

print(results)
    

C:\Users\james\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[(0.6453074694440342, 0.030289891266754906), (0.7408026179542532, 0.07230640343397225), (0.880753910107852, 0.12788508793635547), (0.9045361774366376, 0.18589163937044476), (0.8725731147293594, 0.23046812666177083), (0.8804581477884486, 0.303063523968698), (0.9118929760121707, 0.42868976504411627), (0.9358792477119952, 0.5791343360752427), (0.9255172066859174, 0.6089652693181459), (0.9402450731641845, 0.6867555912233462), (0.9275323253491959, 0.7048016396899519), (0.9269576766576185, 0.7534569727164189), (0.9482184370146495, 0.8236000224108726), (0.947307368642889, 0.8362086460385737), (0.9441940102948777, 0.8550183485198833), (0.9588963126675999, 0.8876435932472682), (0.9370677764433206, 0.8728318491431577), (0.9396372354744007, 0.8801152357105463)]


# Binary classifiers

In [9]:
# testing model 1

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in lowValence + highValence]
Y = [element[1] for element in lowValence + highValence]
orderedIndexes = [element[2] for element in lowValence + highValence]
results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model1.fit(Xtrain, Ytrain)

    Ypred = model1.predict_proba(Xtest)
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

[(0.35248124207561865, 0.32236732733620244), (0.3464191050221404, 0.31990153311492026), (0.08439249387982213, 0.014143455044017896), (0.0025662438335894208, 0.0025570189047436007), (0.23890662085407088, 0.16145929124763667), (0.22606193792146684, 0.21254576882329976), (0.1235684459474347, 0.12354615308147432), (0.2784777486736316, 0.2715007125596381), (0.30858742101987735, 0.2636234210765456), (0.23399367345681363, 0.2339927915306134), (0.4797099633179626, 0.4683891354353028), (0.30245231607629414, 0.3024523160762943), (0.4426148812658872, 0.43827160493827155), (0.08556272429344024, 0.0853607674127624), (0.25733612751674256, 0.12839386082387716), (0.22865915526377675, 0.1726563170083198), (0.3252233255262385, 0.32507397148460676), (0.37420071569014346, 0.35704629434914525)]


In [ ]:
# testing model 2

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in lowArousal + highArousal]
Y = [element[1] for element in lowArousal + highArousal]
orderedIndexes = [element[2] for element in lowArousal + highArousal]
results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model2.fit(Xtrain, Ytrain)

    Ypred = model2.predict_proba(Xtest)
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

C:\Users\james\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
# testing model 3

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in lowValence + highValence]
Y = [element[1] for element in lowValence + highValence]
orderedIndexes = [element[2] for element in lowValence + highValence]
results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    Xtrain, Ytrain = torch.tensor(Xtrain, dtype = torch.float).unsqueeze(1), torch.tensor(Ytrain, dtype = torch.long)
    Xtest, Ytest = torch.tensor(Xtest, dtype = torch.float).unsqueeze(1), torch.tensor(Ytest, dtype = torch.float).unsqueeze(1)

    optimizer = optim.Adam(model3.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(10):
        optimizer.zero_grad()
        outputs = model3(Xtrain)
        loss = criterion(outputs, Ytrain)
        loss.backward()
        optimizer.step()

    model3.eval()
    with torch.no_grad():
        probs = torch.softmax(model3(Xtest), dim=1)
        expected = np.dot(probs, np.arange(2))

        probs = probs.argmax(axis=1)




        results.append((pearsonr(np.array(probs).ravel(), np.array(expected).ravel())[0], CCcoefficient(probs, expected)))

print(results)
    

[(0.6257000208862121, 0.027535735543228898), (0.8203304464156355, 0.06676152725065121), (0.8121089452417437, 0.08460398091771376), (0.8024438850115085, 0.10753450653559884), (0.8106089758853589, 0.1620385384580633), (0.8390356533492528, 0.22624807192595447), (0.8427277643978976, 0.29468098594903935), (0.835665535871119, 0.31818971916234484), (0.8480114121936048, 0.36712012357646046), (0.8905876127772635, 0.4754256881500325), (0.8713166498881838, 0.5301426900941864), (0.884791180089697, 0.5599321694078341), (0.8739672178386206, 0.6263901011904609), (0.8824255566570106, 0.6616006961632844), (0.8510242191292787, 0.6020024389982345), (0.8826472755586644, 0.6625224736368153), (0.8988491704643414, 0.723772598863536), (0.904349621113684, 0.7680053565381069)]


: 